# Question 1 — Ethena loop mechanics, funding source, max leverage, ROE

**Question:** How does the Ethena loop work, and who is funding it? Trace the yield to its ultimate economic source. What is the maximum leverage achievable under current E-mode parameters, and the resulting ROE?

Full discovery methodology (market identity verification, E-mode category scan, address-based classification of collateral assets) lives in `00_setup_data_sourcing.ipynb`. This notebook pulls the result directly and runs the calculations.

## 1.1 — How Ethena generates USDe's yield

Ethena issues USDe, a synthetic dollar, backed by a portfolio of yield-generating strategies. As of the most recent disclosed breakdown ([Ethena reserve diversification announcement, 2026-04-07](https://unchainedcrypto.com/ethena-overhauls-usde-reserves-with-institutional-lending-and-real-world-assets/)):

| Source | Share / status |
|---|---|
| Delta-neutral basis trade (perpetual futures funding) | **11%** of backing |
| Stablecoin reserves + DeFi lending positions | remainder (~89%, exact split not disclosed) |
| Institutional lending (Anchorage Digital, Maple Institutional, Coinbase Asset Management) | expanding, no % disclosed yet |
| Real-world assets (tokenized T-bills, CLOs, investment-grade corporate bonds, structured credit) | expanding, no % disclosed yet |
| Equity/commodity basis trades, prime lending to trading firms | newly added categories, no % disclosed |
| Reserve Fund (loss-absorbing buffer) | ~$80M+ (early 2026) |

**Flagged assumption:** this is the most recent *disclosed* composition (April 2026), not a live daily breakdown — Ethena doesn't publish real-time portfolio weights at this granularity. The direction is clear and worth noting on its own: perpetual funding, once described as the "primary engine," is now a minority (11%) of backing.

**The basis trade mechanic:** Ethena holds a long spot/staking position (e.g. staked ETH) and an equal-notional short perpetual futures position, delta-hedging USDe's dollar value. In a structurally long-biased perp market, longs pay funding to shorts — so Ethena, as the short, collects funding from leveraged long speculators. This flips in bearish/crowded-short regimes, when funding can go negative.

**sUSDe mechanics:** USDe holders stake into an ERC-4626 vault and receive sUSDe, a share of that vault. As the vault's USDe balance grows from the portfolio's yield, each sUSDe share is redeemable for more USDe — this is how yield reaches holders, as a rising exchange rate rather than a rebase or direct payment.

## 1.2 — The loop

A looper: deposit sUSDe as collateral on Aave (E-mode) → borrow USDe against it → stake the borrowed USDe into sUSDe via Ethena → deposit that new sUSDe back onto Aave → repeat. Each pass adds a shrinking increment, converging to a fixed maximum position (derived in 1.4) rather than growing forever.

At today's rates — sUSDe yield 4.0%, USDe borrow APY 3.2% (assignment's case inputs; Section 0 of the setup notebook confirms the live on-chain rate is 3.298%, closely matching) — the looper earns the 4.0% yield on the *full* leveraged sUSDe stack while paying 3.2% only on the *borrowed* portion. As long as the spread is positive, leverage amplifies it.

## 1.3 — Tracing the yield to its ultimate economic source

The yield is not set or determined by Aave — Aave is only the leverage venue. Given the portfolio composition above, it traces to **two** distinct economic sources, not one:

1. **Demand for leveraged perpetuals exposure** (11% of backing): the ultimate payer is the leveraged long side of the ETH perpetual futures market — traders paying funding to hold leveraged long exposure. This portion is not free money: it is a bet, aggregated across the market, that leveraged longs keep paying to be long, and it can go negative in bear/crowded-short regimes.
2. **The cost of USD credit** (the larger and growing remainder — stablecoin reserves, DeFi lending, institutional loans, T-bills/RWA): this is anchored by the Federal Reserve's policy rate, currently **3.50%–3.75%** ([FOMC, held since 2026-06-17](https://www.federalreserve.gov/newsevents/pressreleases/monetary20260617a.htm)). Fed policy sets the risk-free short-term USD rate, which determines both what T-bills yield and what institutional borrowers are willing to pay for USD credit through Ethena's lending arm. This portion of the yield is a credit spread over (or roughly at) the policy rate, not a crypto-native funding payment — its ultimate payer is institutional USD borrowers and, for the T-bill leg, the US Treasury.

**Why this matters for the loop:** as Ethena's backing diversifies away from perp funding toward rate-anchored instruments, sUSDe's yield becomes less exposed to crypto leverage cycles and more directly a spread over the Fed funds rate. At $132/year on a $1,000 position (Section 1.5), the looped ROE (13.2%) sits well above the Fed Funds Rate (3.50–3.75%) — that gap is what attracts looping activity in the first place.

## 1.4 — E-mode parameters

Obtained two ways: from the Aave app frontend, and by reading the Ethereum Core Market Pool contract's state directly at **block 25,682,519** (unix timestamp 1785858143). Full discovery methodology — how category 32 was identified as the correct sUSDe/USDe pairing, market identity verification, and the rejected alternatives — is in `00_setup_data_sourcing.ipynb`.

Contract: `0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2` (Pool proxy) — [Read as Proxy](https://etherscan.io/address/0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2#readProxyContract), `getEModeCategoryCollateralConfig(32)`.

In [26]:
from web3 import Web3
import datetime, json

RPC = "https://eth.drpc.org"  # archive endpoint -- retains state at our pinned block indefinitely
w3 = Web3(Web3.HTTPProvider(RPC))
assert w3.is_connected(), "RPC not reachable"

POOL = w3.to_checksum_address("0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2")
BLOCK_NUMBER = 25682519
BLOCK_TS = datetime.datetime.utcfromtimestamp(1785858143)

ABI = json.loads('''[
 {"name":"getEModeCategoryCollateralConfig","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],
  "outputs":[{"type":"tuple","components":[{"name":"ltv","type":"uint16"},
   {"name":"liquidationThreshold","type":"uint16"},{"name":"liquidationBonus","type":"uint16"}]}]},
 {"name":"getEModeCategoryLabel","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],"outputs":[{"type":"string"}]}
]''')
pool = w3.eth.contract(address=POOL, abi=ABI)

EMODE_ID = 32
cfg = pool.functions.getEModeCategoryCollateralConfig(EMODE_ID).call(block_identifier=BLOCK_NUMBER)
label = pool.functions.getEModeCategoryLabel(EMODE_ID).call(block_identifier=BLOCK_NUMBER)

LTV = cfg[0] / 10000        # bps of 1.0 -> fraction
LT  = cfg[1] / 10000
LIQ_BONUS = (cfg[2] - 10000) / 10000

print(f"Category {EMODE_ID} ({label}) at block {BLOCK_NUMBER}, {BLOCK_TS} UTC:")
print(f"  LTV                 = {LTV:.2%}")
print(f"  Liquidation Threshold = {LT:.2%}")
print(f"  Liquidation Bonus   = {LIQ_BONUS:.2%}")

Category 32 (PTsUSDe5FEB/USDe) at block 25682519, 2026-08-04 15:42:23 UTC:
  LTV                 = 92.00%
  Liquidation Threshold = 94.00%
  Liquidation Bonus   = 2.00%


**What these mean:**

- **LTV (Loan-to-Value), 92%:** the maximum amount you can borrow against your collateral's value at the moment you open the position — $92 borrowed per $100 of sUSDe deposited. This is what sets the leverage ceiling (1.5 below).
- **Liquidation Threshold, 94%:** the collateral ratio at which a position becomes eligible for liquidation. The 2-point gap between LTV and LT (92% → 94%) is the buffer between "maximum you're allowed to borrow" and "the point you actually get liquidated" — opening at the LTV cap leaves zero further buffer.
- **Liquidation Bonus/Penalty, 2%:** the discount a liquidator receives (equivalently, the haircut the borrower takes) when a position is liquidated — the incentive that gets a position closed out before it goes underwater.

## 1.5 — Maximum leverage and ROE

Looping to the LTV cap is a geometric series: each pass supplies collateral, borrows $LTV$ of it, restakes, and resupplies.

$$\text{Max leverage } L = \frac{1}{1 - LTV}$$

At convergence, collateral $C = E \cdot L$ and debt $D = E \cdot (L-1)$, so:

$$\text{ROE} = \frac{C \cdot y_{sUSDe} - D \cdot r_{USDe}}{E} = L \cdot y_{sUSDe} - (L-1) \cdot r_{USDe}$$

using the assignment's case inputs ($y_{sUSDe} = 4.0\%$, $r_{USDe} = 3.2\%$).

In [27]:
EQUITY = 1000.0
USDE_BORROW_APY = 0.032
SUSDE_NET_YIELD = 0.040

L = 1 / (1 - LTV)
C = EQUITY * L
D = EQUITY * (L - 1)
annual_pnl = C * SUSDE_NET_YIELD - D * USDE_BORROW_APY
roe = annual_pnl / EQUITY

print(f"Max leverage           = 1 / (1 - {LTV:.2f}) = {L:.2f}x")
print(f"Collateral (sUSDe)      = ${C:,.2f}")
print(f"Debt (USDe)             = ${D:,.2f}")
print(f"Net annual P&L on $1,000 = ${annual_pnl:,.2f}")
print(f"ROE                     = {roe:.2%}")
print(f"\nFed Funds Rate (upper)  = 3.75%  -- ROE exceeds this by {roe - 0.0375:.2%}, which is the spread that attracts looping activity.")

Max leverage           = 1 / (1 - 0.92) = 12.50x
Collateral (sUSDe)      = $12,500.00
Debt (USDe)             = $11,500.00
Net annual P&L on $1,000 = $132.00
ROE                     = 13.20%

Fed Funds Rate (upper)  = 3.75%  -- ROE exceeds this by 9.45%, which is the spread that attracts looping activity.


## 1.6 — Summary

- **Mechanism:** deposit sUSDe → borrow USDe in E-mode (category 32) → mint more sUSDe → redeposit, converging to $L = 1/(1-LTV)$.
- **E-mode parameters:** LTV 92%, Liquidation Threshold 94%, Liquidation Bonus 2% — verified against the Pool contract at block 25,682,519 and matching the live Aave app.
- **Max leverage: 12.50x.** ROE at max leverage on $1,000: **13.20%** ($132/yr) — well above the Fed Funds Rate (3.50–3.75%), which is the gap that attracts looping activity.
- **Not a free-standing ceiling:** 92% LTV is zero safety margin — a rational looper runs below it. This is a theoretical maximum, not a claim about what's typically deployed.
- **Yield source:** not Aave, and not solely crypto-native funding — increasingly a credit spread anchored to Fed policy as Ethena's backing diversifies (11% perp funding, majority now DeFi lending/institutional loans/RWA). Aave is a leverage venue on a yield it doesn't set and isn't a counterparty to.